<a href="https://colab.research.google.com/github/BridgingAISocietySummerSchools/Data-Science-AI-Python-Course/blob/main/02_advanced_self_learning/07_numpy_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📓 Notebook 7 — NumPy Fundamentals

> **Advanced & Self-Learning Track — Notebook 7 of 10** · **Estimated time:** 35–45 min · **Difficulty:** Beginner / Intermediate
>
> This track is the self-paced deep-dive of the course; the 90-minute live **Introduction Session** lives in `01_introduction/`.

NumPy ("Numerical Python") is the bedrock of the entire Python data-science ecosystem. **Pandas, scikit-learn, TensorFlow, PyTorch, OpenCV** — all of them store data as NumPy arrays under the hood. Mastering NumPy is the single biggest leverage move in the early stages of a data-science journey.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. Explain *why* NumPy exists (speed + expressiveness vs. plain Python lists).
2. Create arrays from scratch and from existing data.
3. Inspect arrays via `shape`, `dtype`, `ndim`, `size` — and dodge the integer/float **dtype trap**.
4. **Index and slice** 1-D and 2-D arrays, including boolean masks.
5. Perform **vectorised** arithmetic and use universal functions (`np.sin`, `np.exp`, …).
6. Understand **broadcasting** — the most important NumPy concept after slicing.
7. Compute statistics over rows / columns with `axis`.
8. Generate reproducible **random data** with `np.random.default_rng`.
9. Reshape, stack, and split arrays.

## ✅ Prerequisites

Notebooks 1–6 of this track, up to `06_functions_modules.ipynb` (lists, comprehensions, functions, imports).

## 1. Why NumPy?

For small datasets pure Python is fine. For *any* serious numerical work, three problems with Python lists become unbearable:

1. **Slow** — Python loops are interpreted; NumPy runs vectorised C code under the hood (often 50–500× faster).
2. **Memory-hungry** — every `int` in a Python list is a full object; a NumPy array of 1 million `int32` uses ~4 MB instead of ~28 MB.
3. **Awkward syntax** — element-wise arithmetic on lists requires comprehensions; NumPy makes it as natural as scalar maths.

Let's see all three in action.

In [1]:
import numpy as np

# Big list vs. big array
py_list = list(range(1_000_000))
np_arr  = np.arange(1_000_000)

import time

# Sum a million numbers, the pure-Python way
t0 = time.perf_counter()
s1 = sum(x*x for x in py_list)
t_py = time.perf_counter() - t0

# Sum a million numbers, NumPy-style (vectorised)
t0 = time.perf_counter()
s2 = (np_arr * np_arr).sum()
t_np = time.perf_counter() - t0

assert s1 == s2
print(f"Pure Python : {t_py*1000:7.1f} ms")
print(f"NumPy       : {t_np*1000:7.1f} ms   ({t_py/t_np:.0f}x faster)")

Pure Python :    23.0 ms
NumPy       :     0.9 ms   (24x faster)


## 2. Creating arrays

There are a handful of constructors you will use constantly:

| Call                        | Result                                    |
|-----------------------------|-------------------------------------------|
| `np.array([1, 2, 3])`       | from a Python list                        |
| `np.zeros((3, 4))`          | shape `(3, 4)`, all zeros                 |
| `np.ones((2, 5))`           | shape `(2, 5)`, all ones                  |
| `np.full((2, 2), 7)`        | filled with 7                             |
| `np.arange(0, 10, 2)`       | like `range`, but as an array              |
| `np.linspace(0, 1, 5)`      | 5 numbers evenly spaced in [0, 1]         |
| `np.eye(3)`                 | 3×3 identity matrix                       |
| `rng.random((rows, cols))`  | random values in [0, 1)                   |

In [2]:
import numpy as np

a = np.array([1, 2, 3, 4, 5])
b = np.zeros((3, 4))
c = np.ones((2, 5))
d = np.full((2, 2), 7)
e = np.arange(0, 10, 2)
f = np.linspace(0, 1, 5)
g = np.eye(3)

for name, arr in [("a", a), ("b", b), ("c", c), ("d", d), ("e", e), ("f", f), ("g", g)]:
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")
    print(arr)
    print()

a: shape=(5,), dtype=int64
[1 2 3 4 5]

b: shape=(3, 4), dtype=float64
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

c: shape=(2, 5), dtype=float64
[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]

d: shape=(2, 2), dtype=int64
[[7 7]
 [7 7]]

e: shape=(5,), dtype=int64
[0 2 4 6 8]

f: shape=(5,), dtype=float64
[0.   0.25 0.5  0.75 1.  ]

g: shape=(3, 3), dtype=float64
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]



## 3. Inspecting an array

The four attributes you check first:

- `shape` — a tuple `(rows, cols, ...)`.
- `dtype` — element data type (`int64`, `float64`, `bool`, …).
- `ndim` — number of dimensions.
- `size` — total number of elements.

In [3]:
A = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])

print(f"A:\n{A}\n")
print(f"shape : {A.shape}")     # (3, 4)
print(f"dtype : {A.dtype}")
print(f"ndim  : {A.ndim}")
print(f"size  : {A.size}")
print(f"max   : {A.max()}")
print(f"min   : {A.min()}")
print(f"mean  : {A.mean()}")
print(f"sum   : {A.sum()}")

A:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

shape : (3, 4)
dtype : int64
ndim  : 2
size  : 12
max   : 12
min   : 1
mean  : 6.5
sum   : 78


> 💡 **In machine learning, `shape` is everything.** Most cryptic errors ("expected (32, 4) got (4, 32)") are shape mismatches. When you debug numerical code, *always* `print(x.shape)` first.

### ⚠️ The dtype trap — integer arrays silently truncate

An array's `dtype` is fixed the moment the array is created. Build it from whole numbers and you get an **integer** array — and any float you later assign into it is **truncated without warning**. No error, no message: the decimals just vanish. This is one of the most common silent bugs in numerical code, so watch it happen once now.

In [4]:
a = np.array([1, 2, 3])            # whole numbers only → integer dtype
print(f"a.dtype = {a.dtype}")

a[0] = 0.99                         # no error — but the 0.99 is TRUNCATED to 0!
print(f"after a[0] = 0.99 → a = {a}   (the .99 silently vanished)")

# Fix 1: ask for floats when you create the array
b = np.array([1, 2, 3], dtype=float)
b[0] = 0.99
print(f"\nb = {b}, dtype = {b.dtype}   (floats survive)")

# Fix 2: a single float in the input upcasts the whole array
c = np.array([1, 2, 3.0])
print(f"np.array([1, 2, 3.0]).dtype = {c.dtype}   (one float upcasts everything)")

# Convert an existing array with .astype — it returns a NEW array
d = a.astype(float)
print(f"a.astype(float) = {d}, dtype = {d.dtype}")

# Good news: division is safe — it always returns floats
print(f"\na / 2 = {a / 2}   (dtype {(a / 2).dtype} — no trap here)")

a.dtype = int64
after a[0] = 0.99 → a = [0 2 3]   (the .99 silently vanished)

b = [0.99 2.   3.  ], dtype = float64   (floats survive)
np.array([1, 2, 3.0]).dtype = float64   (one float upcasts everything)
a.astype(float) = [0. 2. 3.], dtype = float64

a / 2 = [0.  1.  1.5]   (dtype float64 — no trap here)


## 4. Indexing and slicing

For 1-D arrays this is identical to Python lists. For 2-D, the syntax is `A[row, col]` with slicing on each axis.

In [5]:
A = np.arange(1, 21).reshape(4, 5)
print(A)
print()

# Single element
print(f"A[0, 0]   = {A[0, 0]}")
print(f"A[-1, -1] = {A[-1, -1]}")

# Whole row / column
print(f"\nRow 0    : {A[0]}")
print(f"Row 0    : {A[0, :]}    (explicit form)")
print(f"Col 2    : {A[:, 2]}")
print(f"Last col : {A[:, -1]}")

# Sub-matrix
print(f"\nA[1:3, 0:2]:")
print(A[1:3, 0:2])

[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]]

A[0, 0]   = 1
A[-1, -1] = 20

Row 0    : [1 2 3 4 5]
Row 0    : [1 2 3 4 5]    (explicit form)
Col 2    : [ 3  8 13 18]
Last col : [ 5 10 15 20]

A[1:3, 0:2]:
[[ 6  7]
 [11 12]]


### Boolean masking — the most powerful indexing pattern

A boolean array of the same shape can be used to index — only the `True` positions are returned. This is how you do "give me all rows where age > 30" in pure NumPy and (with extra wrapping) in pandas.

In [6]:
x = np.array([10, 25, 7, 18, 33, 2, 41, 9])
print(f"x = {x}")

mask = x > 15
print(f"x > 15 → mask = {mask}")
print(f"x[mask]      = {x[mask]}")

# In one step — the idiomatic form
print(f"x[x > 15]    = {x[x > 15]}")

# Combine masks with & (AND) and | (OR) — parentheses required
print(f"\n10 ≤ x ≤ 30 : {x[(x >= 10) & (x <= 30)]}")

x = [10 25  7 18 33  2 41  9]
x > 15 → mask = [False  True False  True  True False  True False]
x[mask]      = [25 18 33 41]
x[x > 15]    = [25 18 33 41]

10 ≤ x ≤ 30 : [10 25 18]


## 5. Vectorised arithmetic

This is what makes NumPy *feel* like maths: you write the formula once, NumPy applies it to every element. No loops.

In [7]:
temps_c = np.array([0, 10, 20, 25, 30, 35, 100], dtype=float)
print(f"°C : {temps_c}")

# Convert ALL to Fahrenheit in one expression — no loop required
temps_f = temps_c * 9 / 5 + 32
print(f"°F : {temps_f}")

# Element-wise operations between arrays of the same shape
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])
print(f"\na + b = {a + b}")
print(f"a * b = {a * b}")
print(f"b / a = {b / a}")

°C : [  0.  10.  20.  25.  30.  35. 100.]
°F : [ 32.  50.  68.  77.  86.  95. 212.]

a + b = [11 22 33 44]
a * b = [ 10  40  90 160]
b / a = [10. 10. 10. 10.]


### Universal functions (ufuncs)

NumPy provides element-wise versions of every common math function. They are *fast* and they handle entire arrays at once.

In [8]:
x = np.linspace(0, 2 * np.pi, 5)
print(f"x       : {x}")
print(f"sin(x)  : {np.sin(x)}")
print(f"cos(x)  : {np.cos(x)}")
print(f"exp(x)  : {np.exp(x)}")
print(f"log(1+x): {np.log1p(x)}")
print(f"sqrt(x) : {np.sqrt(x)}")

x       : [0.         1.57079633 3.14159265 4.71238898 6.28318531]
sin(x)  : [ 0.0000000e+00  1.0000000e+00  1.2246468e-16 -1.0000000e+00
 -2.4492936e-16]
cos(x)  : [ 1.0000000e+00  6.1232340e-17 -1.0000000e+00 -1.8369702e-16
  1.0000000e+00]
exp(x)  : [  1.           4.81047738  23.14069263 111.31777849 535.49165552]
log(1+x): [0.         0.94421571 1.42108041 1.74263732 1.98556831]
sqrt(x) : [0.         1.25331414 1.77245385 2.17080376 2.50662827]


## 6. Broadcasting — the rule that powers most of NumPy

What if two arrays have *different* shapes? NumPy tries to *stretch* the smaller one to match — this is called **broadcasting**. It is the trickiest idea in this notebook, so we build it up in stages and **print the shapes at every step** (remember: shape is everything).

**Stage 1 — array + scalar.** The scalar is stretched over every element. You already used this without noticing: `temps_c * 9 / 5 + 32` broadcast the scalars 9, 5 and 32 across the whole array.

**Stage 2 — matrix + row vector.** A shape-`(4,)` vector is added to *each row* of a `(3, 4)` matrix.

In [9]:
# Stage 1: array + scalar — the scalar is stretched over every element
a = np.array([1, 2, 3, 4])
print(f"a.shape = {a.shape}")
print(f"a + 100 = {a + 100}   (100 was broadcast to every position)")

# Stage 2: matrix + row vector
A   = np.arange(1, 13).reshape(3, 4)
row = np.array([10, 20, 30, 40])
print(f"\nA.shape   = {A.shape}")
print(f"row.shape = {row.shape}")
print(f"A:\n{A}")
print(f"row: {row}")

result = A + row
print(f"\nA + row (the row was added to EACH of the 3 rows):\n{result}")
print(f"result.shape = {result.shape}")

a.shape = (4,)
a + 100 = [101 102 103 104]   (100 was broadcast to every position)

A.shape   = (3, 4)
row.shape = (4,)
A:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
row: [10 20 30 40]

A + row (the row was added to EACH of the 3 rows):
[[11 22 33 44]
 [15 26 37 48]
 [19 30 41 52]]
result.shape = (3, 4)


**Stage 3 — the general rule.** When NumPy sees two shapes, it lines them up **from the right**. Each pair of dimensions must be **equal**, or one of them must be **1** (missing dimensions on the left count as 1). Whichever side has the 1 gets stretched.

```
A    (3, 4)          A    (3, 4)          A    (3, 4)
row     (4,)  ✔      col  (3, 1)  ✔       v       (3,)  ✘
```

- `row (4,)` is treated as `(1, 4)`: from the right, 4 vs 4 ✔, then 1 vs 3 → stretch ✔ → added to each **row**.
- `col (3, 1)`: from the right, 1 vs 4 → stretch ✔, then 3 vs 3 ✔ → added to each **column**.
- `v (3,)` is treated as `(1, 3)`: from the right, **3 vs 4 — not equal and neither is 1** → error.

The failing case is the important one: a length-3 vector does **not** automatically act as a column of a `(3, 4)` matrix. If that is what you mean, you must *say so* by reshaping it to `(3, 1)`. Run the next cell, then uncomment the failing line and read the error message.

In [10]:
# Stage 3a: column vector + matrix — shape (3, 1) is stretched across the 4 columns
col = np.array([[100], [200], [300]])
print(f"A.shape   = {A.shape}")
print(f"col.shape = {col.shape}")
print(f"A + col (the col was added to EACH of the 4 columns):\n{A + col}")

# Stage 3b: the FAILING case — shapes (3, 4) and (3,) are NOT compatible.
# Uncomment the two lines below, run, and read the error — it names both shapes:
# v = np.array([1, 2, 3])
# A + v      # ValueError: operands could not be broadcast together with shapes (3,4) (3,)

# The fix: turn v into an explicit column of shape (3, 1)
v = np.array([1, 2, 3])
print(f"\nv.shape               = {v.shape}     → incompatible with (3, 4)")
print(f"v.reshape(3, 1).shape = {v.reshape(3, 1).shape}   → compatible")
print(f"A + v.reshape(3, 1):\n{A + v.reshape(3, 1)}")

A.shape   = (3, 4)
col.shape = (3, 1)
A + col (the col was added to EACH of the 4 columns):
[[101 102 103 104]
 [205 206 207 208]
 [309 310 311 312]]

v.shape               = (3,)     → incompatible with (3, 4)
v.reshape(3, 1).shape = (3, 1)   → compatible
A + v.reshape(3, 1):
[[ 2  3  4  5]
 [ 7  8  9 10]
 [12 13 14 15]]


> 🎯 **Mental model.** Line the shapes up from the right; each pair must be equal or contain a 1; the 1 gets stretched. If neither holds, NumPy refuses with `operands could not be broadcast together with shapes ...` — and because the message names both shapes, it is usually all you need to fix the bug.

This single rule replaces what would otherwise be hundreds of nested loops — you will use it in Exercise 7.4 to standardise a whole feature matrix in one line.

## 7. Aggregations along an axis

Sum, mean, max, min — these work over the *whole* array by default, but you can ask for them along a specific **axis**. This is the second concept (after broadcasting) that trips everyone up, so here is the rule worth memorising:

> 🎯 **The axis you name is the dimension that COLLAPSES (disappears).**
>
> For `A.shape == (3, 4)`:
> - `A.sum(axis=0)` → axis 0 (the 3 rows) collapses → result shape `(4,)` → one value per **column**.
> - `A.sum(axis=1)` → axis 1 (the 4 columns) collapses → result shape `(3,)` → one value per **row**.

```
        col 0  col 1  col 2  col 3
row 0     1     2     3     4
row 1     5     6     7     8     ← sum over axis=1 (each row → 1 number)
row 2     9    10    11    12

         ↓  sum over axis=0 (each col → 1 number)
        15   18   21   24
```

The matrix below is small enough to check every number by hand — do it once and the rule sticks.

In [11]:
A = np.arange(1, 13).reshape(3, 4).astype(float)
print(f"A (shape {A.shape}):\n{A}\n")

col_sums = A.sum(axis=0)
row_sums = A.sum(axis=1)
print(f"sum()       : {A.sum()}    (no axis → everything collapses to ONE number)")
print(f"sum(axis=0) : {col_sums}   shape {col_sums.shape} — the 3 rows collapsed → one value per column")
print(f"sum(axis=1) : {row_sums}        shape {row_sums.shape} — the 4 columns collapsed → one value per row")

# Hand-check: the first COLUMN is 1 + 5 + 9 = 15 → matches sum(axis=0)[0]
print(f"\nHand-check: 1 + 5 + 9 = {1 + 5 + 9}, and sum(axis=0)[0] = {col_sums[0]}")

print(f"\nmean(axis=0)   : {A.mean(axis=0)}")
print(f"argmax(axis=1) : {A.argmax(axis=1)}   (index of max in each row — here always the last column)")

A (shape (3, 4)):
[[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]]

sum()       : 78.0    (no axis → everything collapses to ONE number)
sum(axis=0) : [15. 18. 21. 24.]   shape (4,) — the 3 rows collapsed → one value per column
sum(axis=1) : [10. 26. 42.]        shape (3,) — the 4 columns collapsed → one value per row

Hand-check: 1 + 5 + 9 = 15, and sum(axis=0)[0] = 15.0

mean(axis=0)   : [5. 6. 7. 8.]
argmax(axis=1) : [3 3 3]   (index of max in each row — here always the last column)


## 8. Reshape, stack, split

Manipulating shapes is one of the most common operations in ML.

In [12]:
x = np.arange(12)
print(f"x : shape={x.shape} → {x}")

# reshape to whatever you need (must match total size)
m34 = x.reshape(3, 4)
m43 = x.reshape(4, 3)
m26 = x.reshape(2, 6)
print(f"\nreshape (3,4):\n{m34}")
print(f"\nreshape (4,3):\n{m43}")

# Use -1 to let NumPy figure out one dimension
auto = x.reshape(-1, 4)
print(f"\nreshape (-1,4):\n{auto}   shape={auto.shape}")

# Flatten back to 1-D
print(f"\nflatten        : {m34.flatten()}")
print(f"ravel          : {m34.ravel()}    (same, but a view when possible)")

x : shape=(12,) → [ 0  1  2  3  4  5  6  7  8  9 10 11]

reshape (3,4):
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

reshape (4,3):
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]

reshape (-1,4):
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]   shape=(3, 4)

flatten        : [ 0  1  2  3  4  5  6  7  8  9 10 11]
ravel          : [ 0  1  2  3  4  5  6  7  8  9 10 11]    (same, but a view when possible)


In [13]:
# Stacking arrays
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("hstack (horizontal):", np.hstack([a, b]))
print("vstack (vertical) :\n", np.vstack([a, b]))
print("stack new axis    :\n", np.stack([a, b]))   # shape (2, 3)
print("column_stack      :\n", np.column_stack([a, b]))   # shape (3, 2)

hstack (horizontal): [1 2 3 4 5 6]
vstack (vertical) :
 [[1 2 3]
 [4 5 6]]
stack new axis    :
 [[1 2 3]
 [4 5 6]]
column_stack      :
 [[1 4]
 [2 5]
 [3 6]]


## 9. Random data — reproducibly

For experiments and demos you want random data that *isn't actually random*: you want the same numbers every time you run the notebook so results are reproducible. Use `np.random.default_rng(seed)`.

In [14]:
rng = np.random.default_rng(seed=42)

print("rng.random((3, 4)):")
print(rng.random((3, 4)))

print("\nrng.integers(low=0, high=10, size=(2, 5)):")
print(rng.integers(0, 10, size=(2, 5)))

print("\nrng.normal(mean=50, std=15, size=8):")
print(rng.normal(50, 15, size=8).round(2))

print("\nrng.choice(['cat','dog','bird'], size=5):")
print(rng.choice(["cat", "dog", "bird"], size=5))

rng.random((3, 4)):
[[0.77395605 0.43887844 0.85859792 0.69736803]
 [0.09417735 0.97562235 0.7611397  0.78606431]
 [0.12811363 0.45038594 0.37079802 0.92676499]]

rng.integers(low=0, high=10, size=(2, 5)):
[[7 6 4 8 5]
 [4 4 2 0 5]]

rng.normal(mean=50, std=15, size=8):
[35.62 63.18 49.25 47.23 39.79 68.34 47.68 43.58]

rng.choice(['cat','dog','bird'], size=5):
['bird' 'cat' 'dog' 'dog' 'dog']


> 💡 The old `np.random.seed` / `np.random.rand` API still works but the **Generator API** (`default_rng`) is the recommended modern way: explicit, isolated, and supports parallel streams.

## 10. Putting it all together — a tiny applied example

Suppose we have **5 students** each taking **3 exams**. We want to:

1. compute each student's **average** (per-row),
2. each exam's **average** (per-column),
3. apply a **+5 bonus** to the English exam only, then re-compute,
4. find the **top-3 students** by overall average.

This combines indexing, broadcasting, axis aggregation, and `argsort`.

In [15]:
rng = np.random.default_rng(0)

# 5 students × 3 exams (scores 0-100)
scores = rng.integers(40, 100, size=(5, 3))
students = np.array(["Alice", "Bob", "Carol", "Dan", "Eva"])
exams    = np.array(["Math", "Sci", "Eng"])

print(f"      {exams[0]:>4} {exams[1]:>4} {exams[2]:>4}")
for name, row in zip(students, scores):
    print(f"{name:<6}{row[0]:>5}{row[1]:>5}{row[2]:>5}")

# 1. per-student average
student_avg = scores.mean(axis=1)
print("\nStudent averages:")
for name, avg in zip(students, student_avg):
    print(f"  {name:<6} {avg:5.2f}")

# 2. per-exam average
exam_avg = scores.mean(axis=0)
print("\nExam averages:", dict(zip(exams, exam_avg.round(2))))

# 3. +5 bonus on the English exam (column index 2) only — broadcasting [0, 0, 5]
bonus = np.array([0, 0, 5])
new_scores = np.clip(scores + bonus, 0, 100)
print(f"\nAfter +5 English bonus:\n{new_scores}")

# 4. top-3 by overall average
new_avg = new_scores.mean(axis=1)
order = np.argsort(-new_avg)             # descending
print("\nRanking after bonus:")
for rank, idx in enumerate(order, start=1):
    print(f"  {rank}. {students[idx]:<6}  avg = {new_avg[idx]:5.2f}")

      Math  Sci  Eng
Alice    91   78   70
Bob      56   58   42
Carol    44   40   50
Dan      88   78   94
Eva      70   76   98

Student averages:
  Alice  79.67
  Bob    52.00
  Carol  44.67
  Dan    86.67
  Eva    81.33

Exam averages: {np.str_('Math'): np.float64(69.8), np.str_('Sci'): np.float64(66.0), np.str_('Eng'): np.float64(70.8)}

After +5 English bonus:
[[ 91  78  75]
 [ 56  58  47]
 [ 44  40  55]
 [ 88  78  99]
 [ 70  76 100]]

Ranking after bonus:
  1. Dan     avg = 88.33
  2. Eva     avg = 82.00
  3. Alice   avg = 81.33
  4. Bob     avg = 53.67
  5. Carol   avg = 46.33


**What just happened?**

- We used **shape** explicitly: `(5, 3)`. Rows are students, columns are exams.
- `axis=1` gave us per-row (per-student) means; `axis=0` gave us per-column (per-exam) means.
- `bonus = np.array([0, 0, 5])` was **broadcast** across all 5 rows.
- `np.clip` capped the result at 100.
- `np.argsort(-new_avg)` returned the indices that would sort the array in descending order — the ranking.

That's a tiny taste of how NumPy thinks. You will see exactly these patterns in scikit-learn (Notebook 9) where rows are *samples* and columns are *features*.

## 🧪 Practice exercises

### Exercise 7.1 — Create and slice

1. Create a 4×4 array containing the numbers 1–16.
2. Print the second row.
3. Print the third column.
4. Print the **bottom-right 2×2** sub-matrix.

In [16]:
# Your code here  👇
import numpy as np


<details>
<summary>💡 Click to reveal the solution</summary>

```python
A = np.arange(1, 17).reshape(4, 4)
print(A)
print()
print("row 1 :", A[1])
print("col 2 :", A[:, 2])
print("bottom-right 2x2:\n", A[-2:, -2:])
```
</details>

### Exercise 7.2 — Vectorised z-score

Given a 1-D array `x`, compute its **z-score**: subtract the mean, divide by the standard deviation. Verify that the result has mean ≈ 0 and std ≈ 1.

In [17]:
# Your code here  👇
x = np.array([12.0, 18.0, 17.0, 22.0, 25.0, 14.0, 19.0, 21.0])


<details>
<summary>💡 Click to reveal the solution</summary>

```python
z = (x - x.mean()) / x.std()
print("z         :", z.round(3))
print("z.mean()  :", z.mean())
print("z.std()   :", z.std())
```

Standardising features so they have mean 0 and std 1 is a *very* common preprocessing step in ML.
</details>

### Exercise 7.3 — Boolean filtering

Given an array of daily temperatures, print:

1. The number of *hot* days (temperature > 30).
2. The mean temperature of *cold* days (< 10).
3. A new array where any temperature above 40 is clipped to 40.

In [18]:
# Your code here  👇
temps = np.array([5, 25, 35, 12, 28, 42, 3, 19, 31, 8, 22, 45])


<details>
<summary>💡 Click to reveal the solution</summary>

```python
n_hot = (temps > 30).sum()
print(f"Hot days     : {n_hot}")

cold_mean = temps[temps < 10].mean()
print(f"Mean of cold : {cold_mean:.2f}")

clipped = np.clip(temps, None, 40)
print(f"Clipped      : {clipped}")
```

**Pattern.** `(condition).sum()` counts how many entries satisfy a condition — the boolean array of True/False is summed as 1/0.
</details>

### Exercise 7.4 — Broadcasting in action

You have a 10×3 matrix `X` of features and want to **standardise each column** to have mean 0 and std 1 — i.e. subtract the column mean and divide by the column std. Do it in **one expression** using broadcasting (no loops).

In [19]:
# Your code here  👇
rng = np.random.default_rng(0)
X = rng.normal(loc=[5, 100, 0.5], scale=[1, 25, 0.1], size=(10, 3))
print("Original X:\n", X.round(3))
print("\nCol means :", X.mean(axis=0).round(3))
print("Col stds  :", X.std(axis=0).round(3))


Original X:
 [[  5.126  96.697   0.564]
 [  5.105  86.608   0.536]
 [  6.304 123.677   0.43 ]
 [  3.735  84.418   0.504]
 [  2.675  94.53    0.375]
 [  4.268  86.394   0.468]
 [  5.412 126.063   0.487]
 [  6.366  83.37    0.535]
 [  5.903 102.35    0.426]
 [  4.078  88.557   0.522]]

Col means : [ 4.897 97.266  0.485]
Col stds  : [ 1.134 14.921  0.057]


<details>
<summary>💡 Click to reveal the solution</summary>

```python
X_std = (X - X.mean(axis=0)) / X.std(axis=0)
print("Standardised X (rounded):\n", X_std.round(3))
print("\nCheck means :", X_std.mean(axis=0).round(6))
print("Check stds  :", X_std.std(axis=0).round(6))
```

`X.mean(axis=0)` has shape `(3,)`, and broadcasting stretches it across all 10 rows — same for the std. Mean and std of the result should be (almost) 0 and 1 per column.
</details>

### Exercise 7.5 — Debug me 🐞

The cell below should compute the **column means** but the output looks suspicious. Find and fix the bug.

In [20]:
A = np.arange(1, 13).reshape(3, 4)
print("A:\n", A)

means = A.mean(axis=1)         # bug
print("Column means:", means)


A:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
Column means: [ 2.5  6.5 10.5]


<details>
<summary>💡 Click to reveal the solution</summary>

`axis=1` collapses **columns** to produce a value per **row** — i.e. row means, not column means. For column means use `axis=0`.

```python
means = A.mean(axis=0)
print("Column means:", means)
```

The classic mnemonic: `axis=k` means *"this axis disappears"*. Row index is axis 0, column index is axis 1.
</details>

## 🎁 Bonus mini-project — Simulating a fair coin

Use `rng.integers(0, 2, size=n)` to simulate n coin flips (0 = tails, 1 = heads). For `n = 10`, `n = 1_000`, `n = 100_000`:

1. Print the empirical proportion of heads.
2. Show how this converges to 0.5 as n grows (the **law of large numbers**).
3. Bonus: plot the **running proportion** vs flip number.

In [21]:
# Your code here  👇


<details>
<summary>💡 Click to reveal the solution</summary>

```python
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

for n in [10, 1_000, 100_000]:
    flips = rng.integers(0, 2, size=n)
    print(f"n={n:>6}: P(heads) ≈ {flips.mean():.4f}")

# Running proportion plot
n = 5_000
flips = rng.integers(0, 2, size=n)
running = np.cumsum(flips) / np.arange(1, n + 1)

plt.figure(figsize=(8, 4))
plt.axhline(0.5, color="red", ls="--", label="True P=0.5")
plt.plot(running, lw=1.2)
plt.title("Law of large numbers: running proportion of heads")
plt.xlabel("Number of flips")
plt.ylabel("Proportion of heads")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

## 🧠 Key takeaways

1. NumPy arrays are **homogeneous**, **typed**, and **vectorised** — that's why they're fast.
2. Always know your array's `shape` and `dtype`; most ML bugs are shape mismatches — and integer arrays **silently truncate** floats you assign into them.
3. **Indexing / slicing** works on each axis; boolean masks let you filter.
4. **Vectorised arithmetic** + **ufuncs** replace explicit loops — write the formula, not the loop.
5. **Broadcasting** stretches arrays of compatible shapes — no for-loops needed for row/column operations.
6. Aggregate along an axis: `axis=0` collapses rows (one value per column), `axis=1` collapses columns.
7. Use **`np.random.default_rng(seed)`** for reproducible random data.
8. Everything downstream is arrays underneath: a pandas DataFrame is labelled NumPy arrays, and scikit-learn expects a feature matrix of shape `(n_samples, n_features)` — exactly the *rows = samples, columns = features* convention you used here.

## ✅ Self-assessment

- [ ] Build a 2-D array and inspect `shape`, `dtype`, `ndim`, `size`.
- [ ] Slice a sub-matrix and select a column.
- [ ] Filter with a boolean mask.
- [ ] Compute z-scores via broadcasting.
- [ ] Predict which shape pairs broadcast and which raise an error — e.g. `(3, 4) + (4,)` vs `(3, 4) + (3,)`.
- [ ] Explain why `a[0] = 0.99` in an integer array stores `0`.
- [ ] Use `axis=0` vs `axis=1` correctly for aggregations.
- [ ] Reshape and stack arrays.

## 🚀 Next step

Continue with **Notebook 8 — Matplotlib Basics** ([`08_matplotlib_basics.ipynb`](08_matplotlib_basics.ipynb)) to turn these arrays into beautiful, informative plots.